In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# Tb₂Ti₂O₇ on HEiDi

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit the pyrochlore Tb₂Ti₂O₇ to constant-wavelength neutron single-crystal data from HEiDi at
MLZ, with anisotropic displacements and Becker-Coppens extinction.

It builds the catalogue case `sc-neut-cwl_tb2ti2o7-heidi` step by step: the same project the
command line fits and the app's Examples open.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='sc-neut-cwl_tb2ti2o7-heidi')

## Define the structure

Load the structure from EasyDiffraction's collection. Its CIF gives the space group, the cell and
the atom sites.

In [4]:
path = edi.download_data('sc-neut-cwl_tb2ti2o7-heidi-tb2ti2o7', destination='data')
project.structures.add_from_cif_path(path)
structure = project.structures['tb2ti2o7']

Data 'sc-neut-cwl_tb2ti2o7-heidi-tb2ti2o7' is at data/sc-neut-cwl_tb2ti2o7-heidi-tb2ti2o7.edi


In [5]:
project.display.structure('tb2ti2o7')

View(html=<884,874 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('sc-neut-cwl_tb2ti2o7-heidi-heidi-data', destination='data')
project.experiments.add_from_data_path(
    name='heidi',
    data_path=data_path,
    sample_form='single crystal',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['heidi']
experiment.neutron_scattering_length = 'sears1992'
experiment.extinction.type = 'becker-coppens-mixed'
experiment.instrument.name = 'HEiDi'
experiment.instrument.facility = 'MLZ (FRM II)'
experiment.instrument.setup_wavelength = 0.793
experiment.extinction.domain_radius = 25
experiment.extinction.crystal_radius = 3
experiment.linked_structures.create(structure_id='tb2ti2o7', scale=4)

Data 'sc-neut-cwl_tb2ti2o7-heidi-heidi-data' is at data/sc-neut-cwl_tb2ti2o7-heidi-heidi.hkl


In [7]:
project.display.pattern('heidi')

View(html=<105,207 characters>, height=480)

## Perform the analysis

One new group per fit: the scale, O1's position, the displacement tensors, then the extinction.
diffraction-lib also fits occupancies and isotropic displacements, which this case keeps fixed.

### Scale

The scale brings the calculated intensities to the measured ones.

In [8]:
experiment.linked_structures['tb2ti2o7'].scale.free = True

In [9]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,heidi,linked_structure,tb2ti2o7,scale,4,0,-inf,inf,


In [10]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 1 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        22.22        75.58
      1      0.003      19.62        58.90   22.1% ↓
  done · converged · 1 iters · reduced χ² 58.90 · wR(F²) 19.62 % · 0.00 s


View(html=<83,457 characters>, height=376)

In [11]:
project.display.pattern('heidi')

View(html=<105,709 characters>, height=480)

### O1's position

The one coordinate the symmetry leaves free.

In [12]:
structure.atom_sites['O1'].fract_x.free = True

In [13]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 2 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        19.62        59.17
      1      0.004      14.74        33.42   43.5% ↓
  done · converged · 1 iters · reduced χ² 33.42 · wR(F²) 14.74 % · 0.00 s


View(html=<84,941 characters>, height=376)

In [14]:
project.display.pattern('heidi')

View(html=<105,793 characters>, height=480)

### Anisotropic displacements

In [15]:
for tensor in structure.atom_site_aniso:
    for component in (
        tensor.adp_11,
        tensor.adp_22,
        tensor.adp_33,
        tensor.adp_12,
        tensor.adp_13,
        tensor.adp_23,
    ):
        component.free = not component.symmetry_constrained

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 10 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                        14.74        34.69
      1      0.005       6.09         5.93   82.9% ↓
  done · converged · 1 iters · reduced χ² 5.93 · wR(F²) 6.09 % · 0.01 s


View(html=<95,124 characters>, height=376)

In [17]:
project.display.pattern('heidi')

View(html=<106,138 characters>, height=480)

### Extinction

The domain radius and the mosaic spread.

In [18]:
experiment.extinction.domain_radius.free = True
experiment.extinction.mosaic_spread.free = True

In [19]:
project.analysis.fit()
project.display.fit.results()

edi fit: project sc-neut-cwl_tb2ti2o7-heidi, 220 pts fitted / 220 loaded, 12 free
   iter   time (s) wR(F²) (%)  reduced χ²   change
                         6.09         5.98
      1      0.005       2.90         1.35   77.4% ↓
      2      0.006       2.88         1.33   1.6% ↓
      3      0.007       2.88         1.33   
  done · converged · 3 iters · reduced χ² 1.33 · wR(F²) 2.88 % · 0.01 s


View(html=<98,463 characters>, height=376)

In [20]:
project.display.pattern('heidi')

View(html=<106,154 characters>, height=480)

In [21]:
project.display.fit.correlations()

,1,2,3,4,5
1 tb2ti2o7.atom_site_aniso.Tb.adp_11,,,,,
2 tb2ti2o7.atom_site_aniso.O1.adp_22,0.53,,,,
3 heidi.linked_structure.tb2ti2o7.scale,0.69,0.65,,,
4 heidi.extinction.domain_radius,0.20,0.21,0.19,,
5 heidi.extinction.mosaic_spread,0.20,0.15,0.49,-0.63,


In [22]:
project.display.structure('tb2ti2o7')

View(html=<900,740 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [23]:
project.report.save_html('sc-neut-cwl_tb2ti2o7-heidi.html')

Saving report to 'sc-neut-cwl_tb2ti2o7-heidi.html'


## Save the project

In [24]:
project.save_as('sc-neut-cwl_tb2ti2o7-heidi')

Saving project to 'sc-neut-cwl_tb2ti2o7-heidi'
